# Sistema de Pareamento e Clasificação de Cidades

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

In [2]:
df = pd.read_csv("features_ibge_municipios_2016_2026.csv")

In [3]:
# 2. Filtrar o ano mais recente para análise
ANOS_ANALISE = [2016, 2017, 2018, 2019, 2020, 2021]
df_intervalo = df[df['ano'].isin(ANOS_ANALISE)].copy()

In [4]:
features_cidade = [
    'populacao',
    'pib_per_capita',
    'pct_va_servicos',
    'pct_va_industria',
    'pct_va_agropecuaria',
    'pct_impostos'
]

In [5]:
df_agrupado = df_intervalo.groupby(['id_municipio', 'nome_municipio', 'sigla_uf', 'regiao'])[features_cidade].mean().reset_index()

In [6]:
def adicionar_cidades_similares(df_input, features, modo_regiao='mesma', n_similares=3):
    """
    Adiciona colunas com os IDs, nomes e % de similaridade dos municípios mais próximos.
    
    Novas colunas geradas para cada similar (i de 1 a n_similares):
      - id_municipio_similar_{i}
      - municipio_similar_{i}
      - percentual_similaridade_{i}
    """
    df_resultado = df_input.copy()

    # Inicializar as novas colunas
    for i in range(1, n_similares + 1):
        df_resultado[f'id_municipio_similar_{i}'] = None
        df_resultado[f'municipio_similar_{i}'] = None
        df_resultado[f'percentual_similaridade_{i}'] = np.nan

    # Definir os grupos de execução (Por região ou Brasil inteiro)
    if modo_regiao == 'mesma':
        grupos = df_resultado.groupby('regiao')
    else:
        grupos = [('brasil', df_resultado)]

    # Processar grupo a grupo
    for _, grupo_df in grupos:
        if len(grupo_df) <= 1:
            continue

        # Normalização Z-Score das variáveis dentro do grupo
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(grupo_df[features])

        # Treinamento do KNN
        k = min(len(grupo_df), n_similares + 1)
        knn = NearestNeighbors(n_neighbors=k, metric='euclidean')
        knn.fit(X_scaled)

        # Encontrar os vizinhos mais próximos
        distances, indices = knn.kneighbors(X_scaled)

        # Preencher os resultados na tabela
        for idx_local, (dists, idxs) in enumerate(zip(distances, indices)):
            idx_global = grupo_df.index[idx_local]

            # Ignora o índice 0 (a própria cidade)
            sim_count = 1
            for d, idx_vizinho in zip(dists[1:], idxs[1:]):
                # Dados da cidade similar
                id_sim = grupo_df.iloc[idx_vizinho]['id_municipio']
                cidade_sim = grupo_df.iloc[idx_vizinho]['nome_municipio']
                uf_sim = grupo_df.iloc[idx_vizinho]['sigla_uf']
                nome_formatado = f"{cidade_sim} ({uf_sim})"

                # Percentual de similaridade (Exponencial da distância Z-Score)
                sim_pct = round(np.exp(-d / 2.0) * 100, 2)

                # Atribuição dos valores nas colunas
                df_resultado.at[idx_global, f'id_municipio_similar_{sim_count}'] = int(id_sim)
                df_resultado.at[idx_global, f'municipio_similar_{sim_count}'] = nome_formatado
                df_resultado.at[idx_global, f'percentual_similaridade_{sim_count}'] = sim_pct
                
                sim_count += 1

    return df_resultado

In [9]:
df_final = adicionar_cidades_similares(df_agrupado, features_cidade, modo_regiao='mesma', n_similares=3)

In [10]:
df_final.to_csv("municipios_com_cidades_similares.csv", index=False, encoding="utf-8-sig")

In [11]:
df_similar = pd.read_csv('municipios_com_cidades_similares.csv')

In [12]:
df_similar[df_similar['nome_municipio'] =='Hortolândia']

,id_municipio,nome_municipio,sigla_uf,regiao,populacao,pib_per_capita,pct_va_servicos,pct_va_industria,pct_va_agropecuaria,pct_impostos,id_municipio_similar_1,municipio_similar_1,percentual_similaridade_1,id_municipio_similar_2,municipio_similar_2,percentual_similaridade_2,id_municipio_similar_3,municipio_similar_3,percentual_similaridade_3
3484,3519071,Hortolândia,SP,Sudeste,228543.0,62702.456667,0.45325,0.298417,0.0001,0.181017,3524402,Jacareí (SP),78.56,3554102,Taubaté (SP),73.48,3520509,Indaiatuba (SP),72.86
